# 05 - Tokenizer cost analysis


## Goal

Run the offline tokenizer analyzer over a canonical list of KDE terms and phrases, find the worst-tokenized terms, and reason about their impact on context budget.


## Prerequisites

- No model downloads. The default tokenizer is a pure-Python whitespace fallback so everything runs offline.
- If you have `transformers` installed locally you can substitute a real HF tokenizer in the optional cell at the bottom.


## Environment bootstrap

This cell makes the notebook portable between a local checkout and Colab.

- **Local**: when the notebook lives inside the repo, we add the repo root to `sys.path`
  so the `src` package imports cleanly.
- **Colab**: the import will fail with `ModuleNotFoundError`. We catch that and print a
  one-line reminder showing the `git clone` the learner should run. We deliberately do
  **not** execute the clone for them — the lab policy is *recipes only, no auto-downloads*.


In [ ]:
import sys
from pathlib import Path

try:
    # Local checkout: walk up from the notebook to the repo root.
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "common" / "paths.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            break
    from src.common.paths import REPO_ROOT, MINI_REPO, ensure_dirs
    ensure_dirs()
    print(f"repo root: {REPO_ROOT}")
    print(f"mini repo: {MINI_REPO}")
except ModuleNotFoundError:
    print("`src` not importable. If you are on Colab, run this in a separate cell:")
    print("    !git clone https://example.invalid/kde_ontology_slm_lab.git")
    print("    %cd kde_ontology_slm_lab")
    print("Then re-run this cell. We will not auto-clone for you (lab policy:")
    print("recipes only, no auto-downloads).")


## 1. Run the analyzer with the offline fallback

`analyze()` (in `src/tokenizer/analyze_tokens.py`) measures tokens-per-term and the compression ratio (chars / tokens; higher is better) for the canonical `KDE_TERMS` and `KDE_PHRASES` lists.


In [ ]:
from src.tokenizer.analyze_tokens import analyze, KDE_TERMS, KDE_PHRASES, worst_terms

rep = analyze()
summary = rep.summary()
print(f"tokenizer       : {rep.tokenizer_name}")
print(f"mean compression: {summary['mean_compression']:.2f} chars/token")
print(f"mean tokens/term: {summary['mean_tokens_per_term']:.2f}")
print(f"worst term comp : {summary['worst_terms']:.2f}")
print(f"best term comp  : {summary['best_terms']:.2f}")


## 2. Inspect the worst-tokenized terms

The terms with the *lowest* chars/token ratio are the ones the tokenizer is shredding the most. For a KDE-tuned SLM these are the prime candidates for vocabulary extension (see `configs/tokenizer.yaml`).


In [ ]:
wt = worst_terms(rep, n=10)
print('Worst-tokenized:')
for t in wt:
    print(f'  {t.compression:5.2f} chars/tok  ({t.tokens:2d} tok, {t.chars:3d} chr)  {t.term}')


## 3. Inspect the best-tokenized terms

These are short, common, or alphabetic terms that fit in one or two pieces.


In [ ]:
best = sorted(rep.terms + rep.phrases, key=lambda t: t.compression, reverse=True)[:10]
print('Best-tokenized:')
for t in best:
    print(f'  {t.compression:5.2f} chars/tok  ({t.tokens:2d} tok, {t.chars:3d} chr)  {t.term}')


## 4. Impact on context budget

If a phrase like `qmlRegisterType<KFileSearcher>("org.kde.minisearch", 1, 0, ...)` costs N tokens, and a typical retrieved evidence snippet contains 5 such phrases plus source code, a 2048-token context window can hold fewer evidence items than you would naively expect. The cell below estimates how many evidence snippets fit in a budget.


In [ ]:
BUDGET = 2048
OVERHEAD = 256  # system prompt + instruction + answer scaffolding
avg_evidence_chars = 320  # a one-line citation with a code snippet
mean_comp = summary['mean_compression'] or 1.0
approx_tokens_per_evidence = avg_evidence_chars / mean_comp
fits = max(0, (BUDGET - OVERHEAD) // max(1, approx_tokens_per_evidence))
print(f'budget after overhead   : {BUDGET - OVERHEAD} tokens')
print(f'tokens per evidence (~) : {approx_tokens_per_evidence:.1f}')
print(f'evidence items that fit : {fits:.0f}')


## 5. Optional: plug in a real HF tokenizer

If `transformers` is installed and you already have a tokenizer cached locally, you can pass it via the `tokenizer` argument. We do **not** download anything — this cell no-ops on a fresh machine.


In [ ]:
try:
    from transformers import AutoTokenizer  # noqa: F401
    print('transformers available; load your local tokenizer via:')
    print('  tok = AutoTokenizer.from_pretrained("/path/to/local/cache")')
    print('  rep2 = analyze(tokenizer=tok)')
    print('Compare rep2.summary() to the fallback summary above.')
except ImportError:
    print('transformers not installed; staying with the whitespace fallback.')


## Summary

You measured how brutally a naive tokenizer chews up KDE identifiers and reasoned about the resulting context-budget tradeoff. Notebook 06 uses the graph to produce an SFT dataset; once trained, that model should pay the same per-token cost.


## Exercises

1. Add your team's own product-specific terms to a new list and call `analyze(terms=...)`. Are they cheaper or more expensive than KDE's?
2. Compute, for each evidence snippet your RAG pipeline emits, the actual token cost using the fallback. Plot the distribution.
3. If you enabled tokenizer extension, which 16 KDE tokens would you add first?
